# Random Forest Model

In [4]:
import pandas as pd # type: ignore
Data_final = pd.read_csv('/Users/instructorzamora/Documents/3_Maestria_Estadistica_UNINORTE/3_Tercer_Semestre/Machine_Learning/Deteccion_Fraude/Data_final.csv')
Data_final

,D12,D14,D11,D8,TransAmt,D3,D7,dist1,dist2,V209,...,V285,id_01,D13,isFraud,card4_discover,card4_mastercard,card4_visa,card6_credit,card6_debit,card6_debit or credit
0,0.0,0.0,13.0,37.875,68.500000,13.0,0.0,19.0,37.0,0.0,...,0.0,-5.0,0.0,0.0,1,0,0,1,0,0
1,0.0,0.0,43.0,37.875,29.000000,8.0,0.0,8.0,37.0,0.0,...,0.0,-5.0,0.0,0.0,0,1,0,1,0,0
2,0.0,0.0,315.0,37.875,59.000000,8.0,0.0,287.0,37.0,0.0,...,0.0,-5.0,0.0,0.0,0,0,1,0,1,0
3,0.0,0.0,43.0,37.875,50.000000,0.0,0.0,8.0,37.0,0.0,...,10.0,-5.0,0.0,0.0,0,1,0,0,1,0
4,0.0,0.0,43.0,37.875,50.000000,8.0,0.0,8.0,37.0,0.0,...,0.0,0.0,0.0,0.0,0,1,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
590535,0.0,0.0,56.0,37.875,49.000000,30.0,0.0,48.0,37.0,0.0,...,1.0,-5.0,0.0,0.0,0,0,1,0,1,0
590536,0.0,0.0,0.0,37.875,39.500000,8.0,0.0,8.0,37.0,0.0,...,0.0,-5.0,0.0,0.0,0,1,0,0,1,0
590537,0.0,0.0,0.0,37.875,30.950001,8.0,0.0,8.0,37.0,0.0,...,0.0,-5.0,0.0,0.0,0,1,0,0,1,0
590538,0.0,0.0,22.0,37.875,117.000000,0.0,0.0,3.0,37.0,0.0,...,5.0,-5.0,0.0,0.0,0,1,0,0,1,0


El dataset final es un conjunto de datos extenso de detección de fraude con 590,540 registros y 22 columnas, diseñado para un modelo de machine learning que busca identificar transacciones fraudulentas. Contiene variables numéricas como 'TransAmt' (monto de transacción), 'dist1', 'dist2', y códigos como D12, D14, D11, junto con variables categóricas binarias que representan características de tarjetas de pago (como tipos de tarjetas Discover, Mastercard, Visa, y tipos de tarjetas de crédito/débito). La variable objetivo 'isFraud' es binaria (0 o 1), indicando si una transacción es fraudulenta, mientras que la mayoría de las otras variables son numéricas con muchos valores cercanos a cero, sugiriendo un preprocesamiento de datos previo. Este dataset parece estar preparado para entrenar un modelo de clasificación que pueda predecir la probabilidad de fraude en transacciones financieras.

## Metricas Random Forest Model

In [7]:
# ------------------------
# Paso 1: Importar los paquetes necesarios
# ------------------------
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, accuracy_score, f1_score, roc_auc_score
from joblib import dump
from time import time
from skopt import BayesSearchCV
from skopt.space import Integer, Categorical

# ------------------------
# Paso 2: Cargar los datos
# ------------------------
y = Data_final['isFraud']
x = Data_final.drop(columns=['isFraud'])
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.20,random_state=80,stratify=y)

# ------------------------
# Paso 3: Definir el pipeline
# ------------------------
pipe_rf = Pipeline([
    ('scaler', StandardScaler()),  # opcional para Random Forest
    ('rf', RandomForestClassifier(random_state=42))
])

# ------------------------
# Paso 4: Espacio de búsqueda para BayesSearchCV
# ------------------------
search_spaces = {
    'rf__n_estimators': Integer(50, 300),
    'rf__max_depth': Integer(5, 30),
    'rf__min_samples_split': Integer(2, 20),
    'rf__min_samples_leaf': Integer(1, 20),
    'rf__bootstrap': Categorical([True, False]),
    'rf__criterion': Categorical(['gini', 'entropy'])
}

# ------------------------
# Paso 5: Entrenar el modelo con BayesSearchCV
# ------------------------
bayes_rf = BayesSearchCV(
    estimator=pipe_rf,
    search_spaces=search_spaces,
    n_iter=30,
    scoring='roc_auc',
    cv=5,
    n_jobs=-1,
    random_state=42
)

start_time = time()
bayes_rf.fit(x_train, y_train)
training_time_rf = time() - start_time

# Guardar el modelo
dump(bayes_rf, 'bayes_rf.joblib')

# ------------------------
# Paso 6: Hacer predicciones
# ------------------------
y_pred_rf = bayes_rf.best_estimator_.predict(x_test)
y_pred_proba_rf = bayes_rf.best_estimator_.predict_proba(x_test)[:, 1]

# ------------------------
# Paso 7: Calcular métricas
# ------------------------
precision_rf = precision_score(y_test, y_pred_rf, average='weighted')
recall_rf = recall_score(y_test, y_pred_rf, average='weighted')
accuracy_rf = accuracy_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf, average='weighted')
auc_rf = roc_auc_score(y_test, y_pred_proba_rf)

# ------------------------
# Paso 8: Resultados en DataFrame
# ------------------------
resultados_rf = pd.DataFrame({
    'Precision': [f"{precision_rf:.2f}"],
    'Recall': [f"{recall_rf:.2f}"],
    'Accuracy': [f"{accuracy_rf:.2f}"],
    'F1-Score': [f"{f1_rf:.2f}"],
    'AUC': [f"{auc_rf:.2f}"],
    'CPU time (s)': [round(training_time_rf, 2)]
})

# ------------------------
# Paso 9: Mostrar resultados
# ------------------------
print("Métricas para el modelo Random Forest (Bayesian Optimization):")
display(resultados_rf)


Métricas para el modelo Random Forest (Bayesian Optimization):


,Precision,Recall,Accuracy,F1-Score,AUC,CPU time (s)
0,0.97,0.97,0.97,0.97,0.87,23313.97


El modelo Random Forest demuestra un rendimiento excepcional en la detección de fraudes. Con una precisión del 97%, el modelo es extremadamente preciso en la identificación de transacciones fraudulentas. El recall del 97% indica que captura prácticamente todas las transacciones fraudulentas reales, lo cual es crítico en aplicaciones de detección de fraude. La accuracy del 97% confirma su extraordinaria efectividad general en la clasificación de transacciones. El F1-Score de 0.97, prácticamente perfecto, representa un equilibrio ideal entre precisión y recall, demostrando una capacidad casi única para identificar fraudes. El AUC de 0.87 es particularmente impresionante, sugiriendo una excelente capacidad para discriminar entre transacciones fraudulentas y legítimas. El tiempo de CPU de 23,313.97 segundos (aproximadamente 6.5 horas) refleja la complejidad computacional del modelo, típica de algoritmos de ensemble como Random Forest, pero completamente justificada por su rendimiento sobresaliente en la detección de fraudes.